In [1]:
import numpy as np
import time
import sys

from io import UnsupportedOperation
import pickle as pkl


# Add shared location for auxillary functions
sys.path.insert(1, './../../AuxillaryFunctions')

# import personal Functions
from GenerateClassDataFuncs import GenerateEvalData_ClemCafe
from GenerateClassDataFuncs import GenerateEvalData_OREBA
from EvaluationMethodFuncs import KyritEval_PtVsWindow
from EvaluationMethodFuncs import DongEval_PtVsPt
from EvaluationMethodFuncs import TimePoint2Window
from EvaluationMethodFuncs import PrintStats_SingleLine


import csv



print("Finished importing libraries.")


Finished importing libraries.


In [2]:
### Demo of reading a file ###

# My_File = "./FiveFold_CTCResults_v1/Clem_Intake/fold0/'Clemson_p410_c2_grm_std_uni.csv"
My_File = "./FiveFold_CTCResults_v1/THO_Intake/fold0/OREBA-DIS_1116_1_grm_std_uni.csv"



def ReadPredictionsFileForDetections(My_File):
	with open(My_File, newline='') as csvfile:
		spamreader = csv.reader(csvfile, delimiter=',', quotechar='|')
		cnt=0
		Detections=[]
		for row in spamreader:
			cnt+=1
			if (row[4]!='1'): # Fifth Column is either 1 for null action or 2 for intake event
				# print(f"{cnt/8}")
				Detections.append(cnt/7.5)
			#end if
		#end for row
	#end with open(.csv)
	
	return Detections
# end of ReadPredictionsFileForDetections()

In [3]:
##### Define Constants

ResultsFilesAllFolds = [
	# FOLD 0 Files
	[
		'Clemson_p007_c1_grm_std_uni.csv',	'Clemson_p015_c1_grm_std_uni.csv',	'Clemson_p019_c2_grm_std_uni.csv',
		'Clemson_p024_c1_grm_std_uni.csv',	'Clemson_p026_c3_grm_std_uni.csv',	'Clemson_p028_c2_grm_std_uni.csv',
		'Clemson_p034_c1_grm_std_uni.csv',	'Clemson_p036_c1_grm_std_uni.csv',	'Clemson_p039_c1_grm_std_uni.csv',
		'Clemson_p045_c1_grm_std_uni.csv',	'Clemson_p048_c1_grm_std_uni.csv',	'Clemson_p052_c2_grm_std_uni.csv',
		'Clemson_p055_c2_grm_std_uni.csv',	'Clemson_p057_c3_grm_std_uni.csv',	'Clemson_p060_c3_grm_std_uni.csv',
		'Clemson_p062_c2_grm_std_uni.csv',	'Clemson_p066_c1_grm_std_uni.csv',	'Clemson_p068_c1_grm_std_uni.csv',
		'Clemson_p070_c1_grm_std_uni.csv',	'Clemson_p072_c1_grm_std_uni.csv',	'Clemson_p077_c2_grm_std_uni.csv',
		'Clemson_p079_c2_grm_std_uni.csv',	'Clemson_p082_c2_grm_std_uni.csv',	'Clemson_p085_c1_grm_std_uni.csv',
		'Clemson_p088_c2_grm_std_uni.csv',	'Clemson_p092_c1_grm_std_uni.csv',	'Clemson_p098_c1_grm_std_uni.csv',
		'Clemson_p101_c1_grm_std_uni.csv',	'Clemson_p103_c2_grm_std_uni.csv',	'Clemson_p107_c2_grm_std_uni.csv',
		'Clemson_p110_c1_grm_std_uni.csv',	'Clemson_p114_c1_grm_std_uni.csv',	'Clemson_p116_c3_grm_std_uni.csv',
		'Clemson_p118_c1_grm_std_uni.csv',	'Clemson_p120_c2_grm_std_uni.csv',	'Clemson_p123_c2_grm_std_uni.csv',
		'Clemson_p130_c1_grm_std_uni.csv',	'Clemson_p136_c1_grm_std_uni.csv',	'Clemson_p140_c1_grm_std_uni.csv',
		'Clemson_p144_c1_grm_std_uni.csv',	'Clemson_p146_c3_grm_std_uni.csv',	'Clemson_p151_c1_grm_std_uni.csv',
		'Clemson_p157_c2_grm_std_uni.csv',	'Clemson_p160_c2_grm_std_uni.csv',	'Clemson_p165_c1_grm_std_uni.csv',
		'Clemson_p169_c2_grm_std_uni.csv',	'Clemson_p172_c2_grm_std_uni.csv',	'Clemson_p174_c3_grm_std_uni.csv',
		'Clemson_p176_c2_grm_std_uni.csv',	'Clemson_p179_c1_grm_std_uni.csv',	'Clemson_p180_c3_grm_std_uni.csv',
		'Clemson_p184_c2_grm_std_uni.csv',	'Clemson_p187_c2_grm_std_uni.csv',	'Clemson_p190_c1_grm_std_uni.csv',
		'Clemson_p194_c2_grm_std_uni.csv',	'Clemson_p198_c1_grm_std_uni.csv',	'Clemson_p204_c2_grm_std_uni.csv',
		'Clemson_p207_c1_grm_std_uni.csv',	'Clemson_p209_c1_grm_std_uni.csv',	'Clemson_p217_c1_grm_std_uni.csv',
		'Clemson_p218_c3_grm_std_uni.csv',	'Clemson_p220_c2_grm_std_uni.csv',	'Clemson_p229_c1_grm_std_uni.csv',
		'Clemson_p231_c2_grm_std_uni.csv',	'Clemson_p235_c1_grm_std_uni.csv',	'Clemson_p241_c1_grm_std_uni.csv',
		'Clemson_p244_c2_grm_std_uni.csv',	'Clemson_p248_c1_grm_std_uni.csv',	'Clemson_p253_c2_grm_std_uni.csv',
		'Clemson_p260_c1_grm_std_uni.csv',	'Clemson_p263_c1_grm_std_uni.csv',	'Clemson_p265_c2_grm_std_uni.csv',
		'Clemson_p268_c1_grm_std_uni.csv',	'Clemson_p270_c2_grm_std_uni.csv',	'Clemson_p271_c2_grm_std_uni.csv',
		'Clemson_p273_c1_grm_std_uni.csv',	'Clemson_p275_c2_grm_std_uni.csv',	'Clemson_p277_c3_grm_std_uni.csv',
		'Clemson_p279_c2_grm_std_uni.csv',	'Clemson_p281_c3_grm_std_uni.csv',	'Clemson_p285_c2_grm_std_uni.csv',
		'Clemson_p291_c2_grm_std_uni.csv',	'Clemson_p297_c2_grm_std_uni.csv',	'Clemson_p311_c1_grm_std_uni.csv',
		'Clemson_p315_c1_grm_std_uni.csv',	'Clemson_p322_c1_grm_std_uni.csv',	'Clemson_p326_c1_grm_std_uni.csv',
		'Clemson_p331_c1_grm_std_uni.csv',	'Clemson_p334_c1_grm_std_uni.csv',	'Clemson_p337_c2_grm_std_uni.csv',
		'Clemson_p343_c2_grm_std_uni.csv',	'Clemson_p352_c2_grm_std_uni.csv',	'Clemson_p361_c3_grm_std_uni.csv',
		'Clemson_p372_c2_grm_std_uni.csv',	'Clemson_p396_c1_grm_std_uni.csv',	'Clemson_p401_c1_grm_std_uni.csv',
		'Clemson_p410_c2_grm_std_uni.csv'
		],
	
	# FOLD 1 Files
	[
		'Clemson_p005_c1_grm_std_uni.csv',	'Clemson_p011_c1_grm_std_uni.csv',	'Clemson_p016_c1_grm_std_uni.csv',
		'Clemson_p020_c1_grm_std_uni.csv',	'Clemson_p024_c2_grm_std_uni.csv',	'Clemson_p027_c1_grm_std_uni.csv',
		'Clemson_p029_c1_grm_std_uni.csv',	'Clemson_p034_c2_grm_std_uni.csv',	'Clemson_p036_c2_grm_std_uni.csv',
		'Clemson_p042_c1_grm_std_uni.csv',	'Clemson_p045_c2_grm_std_uni.csv',	'Clemson_p050_c1_grm_std_uni.csv',
		'Clemson_p053_c1_grm_std_uni.csv',	'Clemson_p056_c1_grm_std_uni.csv',	'Clemson_p058_c1_grm_std_uni.csv',
		'Clemson_p061_c1_grm_std_uni.csv',	'Clemson_p064_c1_grm_std_uni.csv',	'Clemson_p066_c2_grm_std_uni.csv',
		'Clemson_p068_c2_grm_std_uni.csv',	'Clemson_p070_c2_grm_std_uni.csv',	'Clemson_p074_c1_grm_std_uni.csv',
		'Clemson_p077_c3_grm_std_uni.csv',	'Clemson_p080_c1_grm_std_uni.csv',	'Clemson_p083_c1_grm_std_uni.csv',
		'Clemson_p086_c1_grm_std_uni.csv',	'Clemson_p089_c1_grm_std_uni.csv',	'Clemson_p093_c1_grm_std_uni.csv',
		'Clemson_p098_c2_grm_std_uni.csv',	'Clemson_p101_c2_grm_std_uni.csv',	'Clemson_p104_c1_grm_std_uni.csv',
		'Clemson_p108_c1_grm_std_uni.csv',	'Clemson_p111_c1_grm_std_uni.csv',	'Clemson_p114_c2_grm_std_uni.csv',
		'Clemson_p117_c1_grm_std_uni.csv',	'Clemson_p118_c2_grm_std_uni.csv',	'Clemson_p121_c1_grm_std_uni.csv',
		'Clemson_p125_c1_grm_std_uni.csv',	'Clemson_p131_c1_grm_std_uni.csv',	'Clemson_p137_c1_grm_std_uni.csv',
		'Clemson_p142_c1_grm_std_uni.csv',	'Clemson_p144_c2_grm_std_uni.csv',	'Clemson_p148_c1_grm_std_uni.csv',
		'Clemson_p151_c2_grm_std_uni.csv',	'Clemson_p158_c1_grm_std_uni.csv',	'Clemson_p161_c1_grm_std_uni.csv',
		'Clemson_p165_c2_grm_std_uni.csv',	'Clemson_p170_c1_grm_std_uni.csv',	'Clemson_p173_c1_grm_std_uni.csv',
		'Clemson_p175_c1_grm_std_uni.csv',	'Clemson_p177_c1_grm_std_uni.csv',	'Clemson_p179_c2_grm_std_uni.csv',
		'Clemson_p181_c1_grm_std_uni.csv',	'Clemson_p185_c1_grm_std_uni.csv',	'Clemson_p187_c3_grm_std_uni.csv',
		'Clemson_p190_c2_grm_std_uni.csv',	'Clemson_p194_c3_grm_std_uni.csv',	'Clemson_p199_c1_grm_std_uni.csv',
		'Clemson_p204_c3_grm_std_uni.csv',	'Clemson_p207_c2_grm_std_uni.csv',	'Clemson_p209_c2_grm_std_uni.csv',
		'Clemson_p217_c2_grm_std_uni.csv',	'Clemson_p218_c4_grm_std_uni.csv',	'Clemson_p221_c1_grm_std_uni.csv',
		'Clemson_p229_c2_grm_std_uni.csv',	'Clemson_p232_c1_grm_std_uni.csv',	'Clemson_p236_c1_grm_std_uni.csv',
		'Clemson_p241_c2_grm_std_uni.csv',	'Clemson_p245_c1_grm_std_uni.csv',	'Clemson_p251_c2_grm_std_uni.csv',
		'Clemson_p256_c1_grm_std_uni.csv',	'Clemson_p260_c2_grm_std_uni.csv',	'Clemson_p263_c2_grm_std_uni.csv',
		'Clemson_p266_c1_grm_std_uni.csv',	'Clemson_p268_c2_grm_std_uni.csv',	'Clemson_p270_c3_grm_std_uni.csv',
		'Clemson_p271_c3_grm_std_uni.csv',	'Clemson_p273_c2_grm_std_uni.csv',	'Clemson_p276_c1_grm_std_uni.csv',
		'Clemson_p278_c1_grm_std_uni.csv',	'Clemson_p279_c3_grm_std_uni.csv',	'Clemson_p282_c1_grm_std_uni.csv',
		'Clemson_p289_c2_grm_std_uni.csv',	'Clemson_p292_c1_grm_std_uni.csv',	'Clemson_p298_c1_grm_std_uni.csv',
		'Clemson_p311_c2_grm_std_uni.csv',	'Clemson_p315_c2_grm_std_uni.csv',	'Clemson_p322_c2_grm_std_uni.csv',
		'Clemson_p326_c2_grm_std_uni.csv',	'Clemson_p331_c2_grm_std_uni.csv',	'Clemson_p334_c2_grm_std_uni.csv',
		'Clemson_p338_c1_grm_std_uni.csv',	'Clemson_p343_c3_grm_std_uni.csv',	'Clemson_p352_c3_grm_std_uni.csv',
		'Clemson_p368_c1_grm_std_uni.csv',	'Clemson_p377_c1_grm_std_uni.csv',	'Clemson_p396_c2_grm_std_uni.csv',
		'Clemson_p401_c2_grm_std_uni.csv',	'Clemson_p411_c1_grm_std_uni.csv'
	],
	
	# FOLD 2 Files
	[
		'Clemson_p005_c2_grm_std_uni.csv', 'Clemson_p011_c2_grm_std_uni.csv', 'Clemson_p016_c2_grm_std_uni.csv', 'Clemson_p021_c2_grm_std_uni.csv',
		'Clemson_p025_c1_grm_std_uni.csv', 'Clemson_p027_c2_grm_std_uni.csv', 'Clemson_p030_c1_grm_std_uni.csv', 'Clemson_p034_c3_grm_std_uni.csv',
		'Clemson_p037_c1_grm_std_uni.csv', 'Clemson_p042_c2_grm_std_uni.csv', 'Clemson_p045_c3_grm_std_uni.csv', 'Clemson_p051_c1_grm_std_uni.csv',
		'Clemson_p054_c1_grm_std_uni.csv', 'Clemson_p056_c2_grm_std_uni.csv', 'Clemson_p059_c1_grm_std_uni.csv', 'Clemson_p061_c2_grm_std_uni.csv',
		'Clemson_p064_c2_grm_std_uni.csv', 'Clemson_p066_c3_grm_std_uni.csv', 'Clemson_p069_c1_grm_std_uni.csv', 'Clemson_p070_c3_grm_std_uni.csv',
		'Clemson_p074_c2_grm_std_uni.csv', 'Clemson_p078_c1_grm_std_uni.csv', 'Clemson_p080_c2_grm_std_uni.csv', 'Clemson_p083_c2_grm_std_uni.csv',
		'Clemson_p087_c1_grm_std_uni.csv', 'Clemson_p090_c1_grm_std_uni.csv', 'Clemson_p093_c2_grm_std_uni.csv', 'Clemson_p099_c1_grm_std_uni.csv',
		'Clemson_p102_c1_grm_std_uni.csv', 'Clemson_p105_c1_grm_std_uni.csv', 'Clemson_p108_c2_grm_std_uni.csv', 'Clemson_p111_c2_grm_std_uni.csv',
		'Clemson_p115_c1_grm_std_uni.csv', 'Clemson_p117_c2_grm_std_uni.csv', 'Clemson_p119_c1_grm_std_uni.csv', 'Clemson_p121_c2_grm_std_uni.csv',
		'Clemson_p125_c2_grm_std_uni.csv', 'Clemson_p132_c1_grm_std_uni.csv', 'Clemson_p137_c2_grm_std_uni.csv', 'Clemson_p142_c2_grm_std_uni.csv',
		'Clemson_p145_c1_grm_std_uni.csv', 'Clemson_p148_c2_grm_std_uni.csv', 'Clemson_p153_c1_grm_std_uni.csv', 'Clemson_p158_c2_grm_std_uni.csv',
		'Clemson_p161_c2_grm_std_uni.csv', 'Clemson_p166_c1_grm_std_uni.csv', 'Clemson_p171_c1_grm_std_uni.csv', 'Clemson_p173_c2_grm_std_uni.csv',
		'Clemson_p175_c2_grm_std_uni.csv', 'Clemson_p177_c2_grm_std_uni.csv', 'Clemson_p179_c3_grm_std_uni.csv', 'Clemson_p182_c1_grm_std_uni.csv',
		'Clemson_p186_c1_grm_std_uni.csv', 'Clemson_p188_c1_grm_std_uni.csv', 'Clemson_p192_c1_grm_std_uni.csv', 'Clemson_p195_c1_grm_std_uni.csv',
		'Clemson_p201_c1_grm_std_uni.csv', 'Clemson_p205_c1_grm_std_uni.csv', 'Clemson_p207_c3_grm_std_uni.csv', 'Clemson_p215_c1_grm_std_uni.csv',
		'Clemson_p217_c3_grm_std_uni.csv', 'Clemson_p219_c1_grm_std_uni.csv', 'Clemson_p224_c1_grm_std_uni.csv', 'Clemson_p230_c1_grm_std_uni.csv',
		'Clemson_p233_c1_grm_std_uni.csv', 'Clemson_p236_c2_grm_std_uni.csv', 'Clemson_p242_c1_grm_std_uni.csv', 'Clemson_p246_c1_grm_std_uni.csv',
		'Clemson_p252_c1_grm_std_uni.csv', 'Clemson_p257_c1_grm_std_uni.csv', 'Clemson_p262_c1_grm_std_uni.csv', 'Clemson_p264_c1_grm_std_uni.csv',
		'Clemson_p266_c2_grm_std_uni.csv', 'Clemson_p269_c1_grm_std_uni.csv', 'Clemson_p270_c4_grm_std_uni.csv', 'Clemson_p272_c1_grm_std_uni.csv',
		'Clemson_p274_c1_grm_std_uni.csv', 'Clemson_p276_c2_grm_std_uni.csv', 'Clemson_p278_c2_grm_std_uni.csv', 'Clemson_p280_c1_grm_std_uni.csv',
		'Clemson_p283_c1_grm_std_uni.csv', 'Clemson_p290_c1_grm_std_uni.csv', 'Clemson_p293_c1_grm_std_uni.csv', 'Clemson_p298_c2_grm_std_uni.csv',
		'Clemson_p311_c3_grm_std_uni.csv', 'Clemson_p318_c1_grm_std_uni.csv', 'Clemson_p322_c3_grm_std_uni.csv', 'Clemson_p329_c1_grm_std_uni.csv',
		'Clemson_p332_c1_grm_std_uni.csv', 'Clemson_p336_c1_grm_std_uni.csv', 'Clemson_p338_c2_grm_std_uni.csv', 'Clemson_p343_c4_grm_std_uni.csv',
		'Clemson_p353_c1_grm_std_uni.csv', 'Clemson_p368_c2_grm_std_uni.csv', 'Clemson_p384_c2_grm_std_uni.csv', 'Clemson_p396_c3_grm_std_uni.csv',
		'Clemson_p406_c1_grm_std_uni.csv', 'Clemson_p411_c2_grm_std_uni.csv'
	],
	
	# FOLD 3 Files
	[
		'Clemson_p006_c1_grm_std_uni.csv', 'Clemson_p012_c2_grm_std_uni.csv', 'Clemson_p017_c2_grm_std_uni.csv', 'Clemson_p022_c1_grm_std_uni.csv',
		'Clemson_p026_c1_grm_std_uni.csv', 'Clemson_p027_c3_grm_std_uni.csv', 'Clemson_p031_c1_grm_std_uni.csv', 'Clemson_p035_c1_grm_std_uni.csv',
		'Clemson_p037_c2_grm_std_uni.csv', 'Clemson_p043_c1_grm_std_uni.csv', 'Clemson_p046_c1_grm_std_uni.csv', 'Clemson_p051_c2_grm_std_uni.csv',
		'Clemson_p054_c2_grm_std_uni.csv', 'Clemson_p057_c1_grm_std_uni.csv', 'Clemson_p060_c1_grm_std_uni.csv', 'Clemson_p061_c3_grm_std_uni.csv',
		'Clemson_p065_c1_grm_std_uni.csv', 'Clemson_p067_c1_grm_std_uni.csv', 'Clemson_p069_c3_grm_std_uni.csv', 'Clemson_p071_c1_grm_std_uni.csv',
		'Clemson_p075_c1_grm_std_uni.csv', 'Clemson_p078_c2_grm_std_uni.csv', 'Clemson_p081_c1_grm_std_uni.csv', 'Clemson_p084_c1_grm_std_uni.csv',
		'Clemson_p087_c2_grm_std_uni.csv', 'Clemson_p090_c2_grm_std_uni.csv', 'Clemson_p095_c1_grm_std_uni.csv', 'Clemson_p099_c2_grm_std_uni.csv',
		'Clemson_p102_c2_grm_std_uni.csv', 'Clemson_p106_c1_grm_std_uni.csv', 'Clemson_p109_c1_grm_std_uni.csv', 'Clemson_p113_c1_grm_std_uni.csv',
		'Clemson_p115_c2_grm_std_uni.csv', 'Clemson_p117_c3_grm_std_uni.csv', 'Clemson_p119_c2_grm_std_uni.csv', 'Clemson_p122_c1_grm_std_uni.csv',
		'Clemson_p129_c1_grm_std_uni.csv', 'Clemson_p132_c2_grm_std_uni.csv', 'Clemson_p138_c1_grm_std_uni.csv', 'Clemson_p143_c1_grm_std_uni.csv',
		'Clemson_p146_c1_grm_std_uni.csv', 'Clemson_p150_c1_grm_std_uni.csv', 'Clemson_p154_c1_grm_std_uni.csv', 'Clemson_p159_c1_grm_std_uni.csv',
		'Clemson_p162_c1_grm_std_uni.csv', 'Clemson_p166_c2_grm_std_uni.csv', 'Clemson_p171_c2_grm_std_uni.csv', 'Clemson_p174_c1_grm_std_uni.csv',
		'Clemson_p175_c3_grm_std_uni.csv', 'Clemson_p178_c1_grm_std_uni.csv', 'Clemson_p180_c1_grm_std_uni.csv', 'Clemson_p182_c2_grm_std_uni.csv',
		'Clemson_p186_c2_grm_std_uni.csv', 'Clemson_p188_c2_grm_std_uni.csv', 'Clemson_p192_c2_grm_std_uni.csv', 'Clemson_p195_c2_grm_std_uni.csv',
		'Clemson_p202_c1_grm_std_uni.csv', 'Clemson_p205_c2_grm_std_uni.csv', 'Clemson_p208_c1_grm_std_uni.csv', 'Clemson_p215_c2_grm_std_uni.csv',
		'Clemson_p218_c1_grm_std_uni.csv', 'Clemson_p219_c2_grm_std_uni.csv', 'Clemson_p226_c1_grm_std_uni.csv', 'Clemson_p230_c2_grm_std_uni.csv',
		'Clemson_p234_c1_grm_std_uni.csv', 'Clemson_p237_c1_grm_std_uni.csv', 'Clemson_p242_c2_grm_std_uni.csv', 'Clemson_p247_c1_grm_std_uni.csv',
		'Clemson_p252_c2_grm_std_uni.csv', 'Clemson_p257_c2_grm_std_uni.csv', 'Clemson_p262_c2_grm_std_uni.csv', 'Clemson_p264_c2_grm_std_uni.csv',
		'Clemson_p267_c1_grm_std_uni.csv', 'Clemson_p269_c2_grm_std_uni.csv', 'Clemson_p270_c5_grm_std_uni.csv', 'Clemson_p272_c2_grm_std_uni.csv',
		'Clemson_p274_c2_grm_std_uni.csv', 'Clemson_p277_c1_grm_std_uni.csv', 'Clemson_p278_c3_grm_std_uni.csv', 'Clemson_p280_c2_grm_std_uni.csv',
		'Clemson_p284_c1_grm_std_uni.csv', 'Clemson_p290_c2_grm_std_uni.csv', 'Clemson_p293_c2_grm_std_uni.csv', 'Clemson_p309_c1_grm_std_uni.csv',
		'Clemson_p312_c1_grm_std_uni.csv', 'Clemson_p320_c1_grm_std_uni.csv', 'Clemson_p324_c1_grm_std_uni.csv', 'Clemson_p329_c2_grm_std_uni.csv',
		'Clemson_p332_c2_grm_std_uni.csv', 'Clemson_p336_c2_grm_std_uni.csv', 'Clemson_p341_c1_grm_std_uni.csv', 'Clemson_p347_c1_grm_std_uni.csv',
		'Clemson_p353_c2_grm_std_uni.csv', 'Clemson_p368_c3_grm_std_uni.csv', 'Clemson_p392_c1_grm_std_uni.csv', 'Clemson_p397_c1_grm_std_uni.csv',
		'Clemson_p406_c2_grm_std_uni.csv', 'Clemson_p413_c1_grm_std_uni.csv'
	],
	
	# FOLD 4 Files
	[
		'Clemson_p006_c2_grm_std_uni.csv', 'Clemson_p013_c1_grm_std_uni.csv', 'Clemson_p019_c1_grm_std_uni.csv', 'Clemson_p023_c1_grm_std_uni.csv',
		'Clemson_p026_c2_grm_std_uni.csv', 'Clemson_p028_c1_grm_std_uni.csv', 'Clemson_p033_c1_grm_std_uni.csv', 'Clemson_p035_c2_grm_std_uni.csv',
		'Clemson_p038_c1_grm_std_uni.csv', 'Clemson_p044_c1_grm_std_uni.csv', 'Clemson_p047_c2_grm_std_uni.csv', 'Clemson_p052_c1_grm_std_uni.csv',
		'Clemson_p055_c1_grm_std_uni.csv', 'Clemson_p057_c2_grm_std_uni.csv', 'Clemson_p060_c2_grm_std_uni.csv', 'Clemson_p062_c1_grm_std_uni.csv',
		'Clemson_p065_c2_grm_std_uni.csv', 'Clemson_p067_c2_grm_std_uni.csv', 'Clemson_p069_c4_grm_std_uni.csv', 'Clemson_p071_c2_grm_std_uni.csv',
		'Clemson_p077_c1_grm_std_uni.csv', 'Clemson_p079_c1_grm_std_uni.csv', 'Clemson_p082_c1_grm_std_uni.csv', 'Clemson_p084_c2_grm_std_uni.csv',
		'Clemson_p088_c1_grm_std_uni.csv', 'Clemson_p091_c1_grm_std_uni.csv', 'Clemson_p096_c1_grm_std_uni.csv', 'Clemson_p100_c1_grm_std_uni.csv',
		'Clemson_p103_c1_grm_std_uni.csv', 'Clemson_p107_c1_grm_std_uni.csv', 'Clemson_p109_c2_grm_std_uni.csv', 'Clemson_p113_c2_grm_std_uni.csv',
		'Clemson_p116_c2_grm_std_uni.csv', 'Clemson_p117_c4_grm_std_uni.csv', 'Clemson_p120_c1_grm_std_uni.csv', 'Clemson_p122_c2_grm_std_uni.csv',
		'Clemson_p129_c2_grm_std_uni.csv', 'Clemson_p133_c1_grm_std_uni.csv', 'Clemson_p139_c1_grm_std_uni.csv', 'Clemson_p143_c2_grm_std_uni.csv',
		'Clemson_p146_c2_grm_std_uni.csv', 'Clemson_p150_c2_grm_std_uni.csv', 'Clemson_p157_c1_grm_std_uni.csv', 'Clemson_p160_c1_grm_std_uni.csv',
		'Clemson_p164_c1_grm_std_uni.csv', 'Clemson_p169_c1_grm_std_uni.csv', 'Clemson_p172_c1_grm_std_uni.csv', 'Clemson_p174_c2_grm_std_uni.csv', 
        'Clemson_p176_c1_grm_std_uni.csv', 'Clemson_p178_c2_grm_std_uni.csv', 'Clemson_p180_c2_grm_std_uni.csv', 'Clemson_p184_c1_grm_std_uni.csv', 
        'Clemson_p187_c1_grm_std_uni.csv', 'Clemson_p189_c1_grm_std_uni.csv', 'Clemson_p194_c1_grm_std_uni.csv', 'Clemson_p195_c3_grm_std_uni.csv', 
        'Clemson_p204_c1_grm_std_uni.csv', 'Clemson_p206_c1_grm_std_uni.csv', 'Clemson_p208_c2_grm_std_uni.csv', 'Clemson_p215_c3_grm_std_uni.csv',
		'Clemson_p218_c2_grm_std_uni.csv', 'Clemson_p220_c1_grm_std_uni.csv', 'Clemson_p226_c2_grm_std_uni.csv', 'Clemson_p231_c1_grm_std_uni.csv',
		'Clemson_p234_c2_grm_std_uni.csv', 'Clemson_p237_c2_grm_std_uni.csv', 'Clemson_p244_c1_grm_std_uni.csv', 'Clemson_p247_c2_grm_std_uni.csv',
		'Clemson_p253_c1_grm_std_uni.csv', 'Clemson_p259_c1_grm_std_uni.csv', 'Clemson_p262_c3_grm_std_uni.csv', 'Clemson_p265_c1_grm_std_uni.csv',
		'Clemson_p267_c2_grm_std_uni.csv', 'Clemson_p270_c1_grm_std_uni.csv', 'Clemson_p271_c1_grm_std_uni.csv', 'Clemson_p272_c3_grm_std_uni.csv',
		'Clemson_p275_c1_grm_std_uni.csv', 'Clemson_p277_c2_grm_std_uni.csv', 'Clemson_p279_c1_grm_std_uni.csv', 'Clemson_p281_c1_grm_std_uni.csv',
		'Clemson_p285_c1_grm_std_uni.csv', 'Clemson_p291_c1_grm_std_uni.csv', 'Clemson_p297_c1_grm_std_uni.csv', 'Clemson_p309_c2_grm_std_uni.csv',
		'Clemson_p312_c2_grm_std_uni.csv', 'Clemson_p320_c2_grm_std_uni.csv', 'Clemson_p324_c2_grm_std_uni.csv', 'Clemson_p329_c3_grm_std_uni.csv',
		'Clemson_p332_c3_grm_std_uni.csv', 'Clemson_p337_c1_grm_std_uni.csv', 'Clemson_p343_c1_grm_std_uni.csv', 'Clemson_p352_c1_grm_std_uni.csv',
		'Clemson_p353_c3_grm_std_uni.csv', 'Clemson_p372_c1_grm_std_uni.csv', 'Clemson_p392_c2_grm_std_uni.csv', 'Clemson_p397_c2_grm_std_uni.csv',
		'Clemson_p410_c1_grm_std_uni.csv'
	]
]

In [4]:
##### Grab GT Values from the Pickle Database

DATABASE_FILEPATH = "./../../Pickle_Databases/ClemDomPkl.pkl"

with open(DATABASE_FILEPATH,'rb') as fh:
	dataset = pkl.load(fh)
#OREBA_Cucumber={'UniqueID','proc_data','handedness','bites_gt'}


# # Grab EvalContent to Have
# STRIDE_sec=5
# CUT_sec=5
# FOLD_INDEX =0
# FOLDS_TOTAL=1
# RR_FLAG = 1
# RESAMPLE_FLAG_FREQ=16

# compiled_eval_data = GenerateEvalData_OREBA(
# 			int(round(CUT_sec*DataFreq)), int(round(STRIDE_sec*DataFreq)),
# 			DATABASE_FILEPATH,
# 			FOLD_INDEX, FOLDS_TOTAL, FOLD_SPLIT=RR_FLAG,
# 			RESAMPLE_FLAG=RESAMPLE_FLAG_FREQ # Frequency of Data Collection in [Hz]
# 			#, RESAMPLE_FLAG=0, SMOOTHING=0 # optional flags not currently used for Paper Experiment
# 			)



print("Finished importing Clemson Pickle.")


Finished importing Clemson Pickle.


In [21]:
# Float value used to determine how many sec the detection can be away from the window
WIN_TOLERANCE = 99
EvalSelect = 1 # 1 = Dong Eval, 2 = Kyritsis Window Tol

# Global Results from all folds (each element is a list from each fold)
All_All_TP = []
All_All_FP = []
All_All_FN = []



for currFold in range(5):

	# Fold number to evaluate. Valid range is [0,4]
	FOLD_SELECT = currFold

	# Directory containing all results for the specified fold
	# PredDirPath = './FiveFold_CTCResults_v1/Clem_Intake/fold{}/'.format(FOLD_SELECT)
	PredDirPath = '/home/jpjolly/CTC_Rouast_Take2/ctc-intake-detection-master/Preds_ClemDom_v4/Fold{}/'.format(FOLD_SELECT)

	# Select Current Fold Results Filenames
	ResultsFiles = ResultsFilesAllFolds[FOLD_SELECT]

	# Create Sanity Check of All Participant and Meal IDs from filenames 
	#     to check against pickle Unique IDs later
	mealID_Check = []
	for filename in ResultsFiles:
		filenameComponents = filename.split("_") 
		mealID_Check.append(filenameComponents[-5] + filenameComponents[-4])
	# end of for filename




	# Define blank sets for results
	All_TP=[]
	All_FP=[]
	All_FN=[]

	for idx in range(0, len(ResultsFiles)):
		currFilename = PredDirPath + ResultsFiles[idx]
		currFileID = mealID_Check[idx]

		# NEED TO SEARCH FOR PICKLE FILE SINCE PICKLE CONTAINS EXTRA MEALS NOT IN CTC FOLD
		try:
			pickle_idx = dataset['UniqueID'].index(currFileID) # get current file index in pickle dataset
		except ValueError:
			print("UNABLE TO FIND PICKLE MEAL CORRESPONDING TO MEAL {}.".format(mealID_Check))
		#end of try statement
		currPickleID = dataset['UniqueID'][pickle_idx]
		
		if currPickleID != currFileID:
			print("ERROR: Mismatched FileID ({}) and MealID ({}).".format(currFileID, currPickleID))
		else:
			print("Matching FileID for {}...".format(currFileID))
		# end of Pickle ID Sanity check

		currDets = np.array(ReadPredictionsFileForDetections(currFilename))
		if EvalSelect==1: # Using Dong Eval
			currGT = dataset['bites_gt'][pickle_idx]/15
		elif EvalSelect==2: # Using Kyritsis Eval with tolerance
			currGT = TimePoint2Window(dataset['bites_gt'][pickle_idx]/15)
		#end EvalSelect switch



		if len(currDets) == 0:
			# print("NO DETECTIONS! {}=FN, mealNum={}".format(len(currGT), meal_num))
			TP = 0
			FP = 0
			FN = len(currGT)
		elif EvalSelect==1: # Using Dong Eval
			[TP, FP, FN, _] = DongEval_PtVsPt(currDets, currGT)
		elif EvalSelect==2: # Using Kyritsis Eval with tolerance
			[TP, FP, FN, FP1, FP2, _] = KyritEval_PtVsWindow(currDets, currGT, WIN_TOLERANCE=WIN_TOLERANCE)
			# [TP, FP, FN, FP1, FP2, Key] = KyritEval_PtVsWindow(currDets, currGT, WIN_TOLERANCE = 0.0, BEFORE_TOL = 0.0, AFTER_TOL = 0.0):
		# end of EvalSelect switch 

		All_TP.append(TP)
		All_FP.append(FP)
		All_FN.append(FN)

		print("\t",end='')
		PrintStats_SingleLine(TP, FP, FN)
	# end of for idx

	All_All_TP.append(All_TP)
	All_All_FP.append(All_FP)
	All_All_FN.append(All_FN)


# end of for currFold





Matching FileID for p007c1...
	84.932 91.176 79.487 31     8      3     
Matching FileID for p015c1...
	77.612 72.222 83.871 26     5      10    
Matching FileID for p019c2...
	96.970 94.118 100.000 16     0      1     
Matching FileID for p024c1...
	94.857 95.402 94.318 83     5      4     
Matching FileID for p026c3...
	83.673 73.214 97.619 41     1      15    
Matching FileID for p028c2...
	84.848 77.778 93.333 14     1      4     
Matching FileID for p034c1...
	88.000 100.000 78.571 22     6      0     
Matching FileID for p036c1...
	88.000 93.617 83.019 44     9      3     
Matching FileID for p039c1...
	91.304 89.362 93.333 42     3      5     
Matching FileID for p045c1...
	90.090 96.154 84.746 50     9      2     
Matching FileID for p048c1...
	93.233 95.385 91.176 62     6      3     
Matching FileID for p052c2...
	62.500 47.619 90.909 20     2      22    
Matching FileID for p055c2...
	94.595 94.595 94.595 35     2      2     
Matching FileID for p057c3...
	68.852 55.263 91.3

In [22]:
#### Caclulate total performance on data

Final_TP = 0
Final_FP = 0
Final_FN = 0

for i in range(5):
	Final_TP += sum(All_All_TP[i]) 
	Final_FP += sum(All_All_FP[i]) 
	Final_FN += sum(All_All_FN[i])
# end of for fold loop 


PrintStats_SingleLine(Final_TP, Final_FP, Final_FN)


89.041 86.882 91.310 17936  1707   2708  


In [ ]:
print(len(currDets))
print(len(currGT))


In [ ]:
currDets

In [6]:
currFilename


'/home/jpjolly/CTC_Rouast_Take2/ctc-intake-detection-master/Preds_ClemDom_v2/Fold0/Clemson_p007_c1_grm_std_uni.csv'